# QA Mini Walkthrough (Single CSV: `predicted_kpis_df.csv`)

โน้ตบุ๊กนี้ใช้ไฟล์เดียว: `outputs/predicted_kpis_df.csv` (ไม่ต้อง join table)

โฟกัสการคำนวณ:
- `active_users_wow_pct`
- `delta_seg` / `wow_pct_seg` (จาก `seg_*_users_prev|delta|wow_pct`)
- `contribution_share` (จาก `target_segments.evidence_metrics.delta_seg`)

เหมาะสำหรับ QA และเลือกดูได้หลายวันจาก snapshot จริงในไฟล์นี้


In [1]:
import json
from pathlib import Path

import pandas as pd

CSV_PATH = Path('../outputs/predicted_kpis_df.csv')
if not CSV_PATH.exists():
    CSV_PATH = Path('outputs/predicted_kpis_df.csv')

# low_memory=False เพื่อให้ schema อ่านตรงขึ้นเวลาไฟล์ใหญ่
kpi_df = pd.read_csv(CSV_PATH, low_memory=False)
kpi_df['brand_id'] = kpi_df['brand_id'].astype(str)
kpi_df['window_size'] = kpi_df['window_size'].astype(str)
kpi_df['window_end_date'] = pd.to_datetime(kpi_df['window_end_date'], utc=True, errors='coerce')

print('Loaded:', CSV_PATH)
print('shape:', kpi_df.shape)
print('brands:', sorted(kpi_df['brand_id'].dropna().unique().tolist()))
print('window_sizes:', sorted(kpi_df['window_size'].dropna().unique().tolist()))


Loaded: ../outputs/predicted_kpis_df.csv
shape: (1632, 574)
brands: ['c-vit', 'see-chan']
window_sizes: ['30d', '60d', '7d', '90d']


In [2]:
kpi_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1632 entries, 0 to 1631
Columns: 574 entries, brand_id to payload
dtypes: bool(78), datetime64[us, UTC](1), float64(478), int64(2), str(15)
memory usage: 32.3 MB


In [8]:
# Print all column names, dtypes, and non-null counts without truncation
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

col_info = pd.DataFrame({
    'column': kpi_df.columns,
    'dtype': kpi_df.dtypes.values,
    'non_null_count': kpi_df.notnull().sum().values,
    'null_count': kpi_df.isnull().sum().values,
    'null_pct': (kpi_df.isnull().sum().values / len(kpi_df) * 100).round(2),
})
print(f'Total columns: {len(kpi_df.columns)}')
print(f'Total rows   : {len(kpi_df)}')
print()
col_info[['column', 'dtype']].reset_index(drop=True).to_csv('column_dtypes.csv', index=False)

Total columns: 574
Total rows   : 1632



In [ ]:
# ดูวันที่ที่เลือกได้ (ปรับ brand/window_size แล้ว rerun cell นี้)
brand_pick = 'c-vit'
window_size_pick = '30d'

available_dates = (
    kpi_df[(kpi_df['brand_id'] == brand_pick) & (kpi_df['window_size'] == window_size_pick)]
    [['window_end_date']]
    .dropna()
    .drop_duplicates()
    .sort_values('window_end_date')
    .reset_index(drop=True)
)

available_dates.tail(20)


## 1) ตั้งค่า row ที่ต้องการ QA

ค่า default ใช้แถวเดียวกับ payload ที่คุณคุยมา:
- `brand_id='c-vit'`
- `window_end_date='2026-02-12'`
- `window_size='30d'`
- `snapshot_freq_days=7`


In [43]:
brand_id = 'c-vit'
window_size = '30d'
window_end_date = pd.Timestamp('2026-02-12T00:00:00Z')
snapshot_freq_days = 7

rows = (
    kpi_df[(kpi_df['brand_id'] == brand_id) & (kpi_df['window_size'] == window_size)]
    .dropna(subset=['window_end_date'])
    .sort_values('window_end_date')
    .reset_index(drop=True)
)
if rows.empty:
    raise ValueError(f'No rows for brand_id={brand_id}, window_size={window_size}')

match_idx = rows.index[rows['window_end_date'] == window_end_date].tolist()
if not match_idx:
    available = rows['window_end_date'].dt.strftime('%Y-%m-%d').tolist()
    raise ValueError(
        f'window_end_date={window_end_date.date()} not found. Available dates (last 20): {available[-20:]}'
    )

idx = match_idx[0]
if idx == 0:
    raise ValueError('Selected row has no previous snapshot in this brand/window_size group.')

cur = rows.iloc[idx]
prev = rows.iloc[idx - 1]

pd.DataFrame([{
    'brand_id': brand_id,
    'window_size': window_size,
    'window_end_date_prev': prev['window_end_date'],
    'window_end_date_now': cur['window_end_date'],
    'snapshot_gap_days': (cur['window_end_date'] - prev['window_end_date']).days,
}])


,brand_id,window_size,window_end_date_prev,window_end_date_now,snapshot_gap_days
0,c-vit,30d,2026-02-05 00:00:00+00:00,2026-02-12 00:00:00+00:00,7


In [44]:
# ช่วงเวลาปัจจุบัน/ช่วงก่อนหน้า (rolling windows)
window_days = int(window_size.replace('d', ''))
end = cur['window_end_date']
prev_end = prev['window_end_date']
start = end - pd.Timedelta(days=window_days - 1)
prev_start = prev_end - pd.Timedelta(days=window_days - 1)

pd.DataFrame([
    {'window': 'current', 'start': start, 'end': end},
    {'window': 'previous (shift by snapshot_freq)', 'start': prev_start, 'end': prev_end},
])


,window,start,end
0,current,2026-01-14 00:00:00+00:00,2026-02-12 00:00:00+00:00
1,previous (shift by snapshot_freq),2026-01-07 00:00:00+00:00,2026-02-05 00:00:00+00:00


## 2) `active_users_wow_pct` คิดยังไง

สูตร:
- `active_users_delta_window = active_users_now - active_users_prev_window`
- `active_users_wow_pct = active_users_delta_window / active_users_prev_window`

ในไฟล์นี้มีทั้งคอลัมน์ base (`active_users`) และคอลัมน์ delta/wow (`*_prev_window`, `*_delta_window`, `*_wow_pct`) ให้เทียบได้ตรงๆ


In [45]:
active_now = float(cur['active_users'])
active_prev_from_row = float(cur['active_users_prev_window']) if pd.notna(cur.get('active_users_prev_window')) else float(prev['active_users'])
active_prev_from_prevrow = float(prev['active_users'])
active_delta_from_row = float(cur['active_users_delta_window']) if pd.notna(cur.get('active_users_delta_window')) else active_now - active_prev_from_prevrow
active_delta_recalc = active_now - active_prev_from_prevrow
active_wow_from_row = float(cur['active_users_wow_pct']) if pd.notna(cur.get('active_users_wow_pct')) else None
active_wow_recalc = active_delta_recalc / active_prev_from_prevrow if active_prev_from_prevrow != 0 else None

pd.DataFrame([{
    'active_users_now': active_now,
    'active_users_prev_window (row)': active_prev_from_row,
    'active_users_prev (previous row)': active_prev_from_prevrow,
    'active_users_delta_window (row)': active_delta_from_row,
    'active_users_delta_window (recalc)': active_delta_recalc,
    'active_users_wow_pct (row)': active_wow_from_row,
    'active_users_wow_pct (recalc)': active_wow_recalc,
}])


,active_users_now,active_users_prev_window (row),active_users_prev (previous row),active_users_delta_window (row),active_users_delta_window (recalc),active_users_wow_pct (row),active_users_wow_pct (recalc)
0,5657.0,6079.0,6079.0,-422.0,-422.0,-0.069419,-0.069419


## 3) Parse `target_segments` จาก CSV string

ใน `predicted_kpis_df.csv` คอลัมน์ `target_segments` เป็น JSON string (ตัวเลขหลายตัวเป็น string)
เราจะ parse + coerce ให้เป็นตัวเลข/boolean ก่อนใช้งาน


In [46]:
def _coerce_json_scalars(x):
    if isinstance(x, list):
        return [_coerce_json_scalars(v) for v in x]
    if isinstance(x, dict):
        return {str(k): _coerce_json_scalars(v) for k, v in x.items()}
    if isinstance(x, str):
        t = x.strip()
        if t == '':
            return x
        if t.lower() == 'true':
            return True
        if t.lower() == 'false':
            return False
        try:
            # พยายามแปลงตัวเลขก่อน
            if any(ch in t for ch in ['.', 'e', 'E']):
                return float(t)
            return int(t)
        except ValueError:
            return x
    return x

def parse_json_field(value, default):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return default
    if isinstance(value, (list, dict)):
        return _coerce_json_scalars(value)
    if isinstance(value, str):
        txt = value.strip()
        if not txt:
            return default
        try:
            return _coerce_json_scalars(json.loads(txt))
        except json.JSONDecodeError:
            return default
    return default

target_segments = parse_json_field(cur.get('target_segments'), default=[])
drivers = parse_json_field(cur.get('drivers'), default=[])

print('drivers:', len(drivers), 'items')
print('target_segments:', len(target_segments), 'items')

active_targets = [t for t in target_segments if isinstance(t, dict) and t.get('metric_family') == 'active_users']
pd.DataFrame([
    {
        'segment_key': t.get('segment_key'),
        'contribution_share': t.get('contribution_share'),
        'delta_seg': (t.get('evidence_metrics') or {}).get('delta_seg'),
        'delta_total': (t.get('evidence_metrics') or {}).get('delta_total'),
        'segment_count_now': (t.get('evidence_metrics') or {}).get('segment_count_now'),
        'segment_share_now': (t.get('evidence_metrics') or {}).get('segment_share_now'),
        'wow_pct_seg': (t.get('evidence_metrics') or {}).get('wow_pct_seg'),
    }
    for t in active_targets
])


drivers: 3 items
target_segments: 2 items


,segment_key,contribution_share,delta_seg,delta_total,segment_count_now,segment_share_now,wow_pct_seg
0,active_0_7d,0.915119,-1035.0,-422.0,1521,0.006701,-0.404930
1,new_users_0_7d,0.084881,-96.0,-422.0,163,0.000718,-0.370656


## 4) `delta_seg` / `wow_pct_seg` มาจากคอลัมน์ไหนในแถวนี้

สำหรับ `metric_family = active_users` ใช้ segment metric = `users`
ดังนั้นสำหรับแต่ละ `segment_key` จะอ่านจาก:
- `seg_<segment_key>_users`
- `seg_<segment_key>_users_prev`
- `seg_<segment_key>_users_delta`
- `seg_<segment_key>_users_wow_pct`


In [ ]:
rows_compare = []
for t in active_targets:
    seg_key = t.get('segment_key')
    if not seg_key:
        continue
    users_col = f'seg_{seg_key}_users'
    prev_col = f'{users_col}_prev'
    delta_col = f'{users_col}_delta'
    wow_col = f'{users_col}_wow_pct'

    if users_col not in cur.index:
        continue

    seg_now = float(cur.get(users_col, 0.0) or 0.0)
    seg_prev = float(cur.get(prev_col, 0.0) or 0.0) if pd.notna(cur.get(prev_col)) else None
    seg_delta_row = float(cur.get(delta_col, 0.0) or 0.0) if pd.notna(cur.get(delta_col)) else None
    seg_wow_row = float(cur.get(wow_col)) if pd.notna(cur.get(wow_col)) else None

    seg_delta_recalc = (seg_now - seg_prev) if seg_prev is not None else None
    seg_wow_recalc = (seg_delta_recalc / seg_prev) if (seg_prev not in (None, 0)) else None

    ev = t.get('evidence_metrics') or {}
    rows_compare.append({
        'segment_key': seg_key,
        'segment_count_now (row)': seg_now,
        'segment_count_now (target_segments)': ev.get('segment_count_now'),
        'seg_prev (row)': seg_prev,
        'delta_seg (row)': seg_delta_row,
        'delta_seg (recalc)': seg_delta_recalc,
        'delta_seg (target_segments)': ev.get('delta_seg'),
        'wow_pct_seg (row)': seg_wow_row,
        'wow_pct_seg (recalc)': seg_wow_recalc,
        'wow_pct_seg (target_segments)': ev.get('wow_pct_seg'),
    })

compare_df = pd.DataFrame(rows_compare)
compare_df


## 5) `contribution_share` คิดยังไง (จาก `target_segments`)

สูตร (ใน metric family เดียวกัน):
- `contribution_share = abs(delta_seg) / sum(abs(delta_seg))`

ในตัวอย่างนี้จะคำนวณใหม่จาก `active_users` target segments แล้วเทียบกับค่าใน `target_segments`


In [ ]:
active_deltas = [float((t.get('evidence_metrics') or {}).get('delta_seg', 0.0) or 0.0) for t in active_targets]
denom = sum(abs(x) for x in active_deltas)

contrib_rows = []
for t in active_targets:
    ev = t.get('evidence_metrics') or {}
    delta_seg = float(ev.get('delta_seg', 0.0) or 0.0)
    contrib_rows.append({
        'segment_key': t.get('segment_key'),
        'delta_seg': delta_seg,
        'delta_total': float(ev.get('delta_total', 0.0) or 0.0),
        'contribution_share (target_segments)': float(t.get('contribution_share', 0.0) or 0.0),
        'contribution_share (recalc)': (abs(delta_seg) / denom) if denom else None,
    })

contrib_df = pd.DataFrame(contrib_rows).sort_values('contribution_share (recalc)', ascending=False)
contrib_df


In [ ]:
# ทำไม contribution_share ไม่ใช่ % ของ delta_total แบบ exact decomposition
summary = pd.DataFrame([{
    'active_users_delta_window (net total change)': float(cur.get('active_users_delta_window', 0.0) or 0.0),
    'sum(delta_seg) across shown active target_segments': sum(active_deltas),
    'sum(abs(delta_seg)) denominator for contribution_share': denom,
    'sum(contribution_share)': float(contrib_df['contribution_share (target_segments)'].sum()) if not contrib_df.empty else 0.0,
}])
summary


## สรุปการอ่านค่า (ใช้กับวันอื่นได้เหมือนกัน)

- `active_users_wow_pct`: net % change ของ total active users เทียบ snapshot ก่อนหน้า
- `delta_seg`: change ของจำนวน user ใน segment นั้นเทียบ snapshot ก่อนหน้า
- `wow_pct_seg`: % change ของจำนวน user ใน segment นั้น
- `contribution_share`: น้ำหนัก normalized ของ `delta_seg` ภายใน active-user target segments ที่ระบบเลือกมาอธิบาย (ไม่ใช่ exact decomposition ของ total)

เปลี่ยนวันได้โดยแก้ cell ตั้งค่า (`brand_id`, `window_size`, `window_end_date`) แล้ว rerun ต่อจากนั้น
